# Velnor Qwen2.5-Coder-1.5B LoRA training
This notebook runs the full 138-example training pool and 20-example held-out set on an allocated CUDA GPU. It saves the adapter, report, logs, and SHA-256 manifest to Google Drive. It does not upload model weights to GitHub.

In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU) before running. GPU allocation is subject to Colab availability.

In [ ]:
import torch, os, subprocess, sys
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is attached. Select a GPU runtime in Colab and rerun this cell; training is intentionally blocked on CPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('PyTorch:', torch.__version__)
print('BF16 supported:', torch.cuda.is_bf16_supported())

In [ ]:
REPO = '/content/velnor'
BRANCH = 'fix/training-provenance-and-evaluation'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--branch',BRANCH,'--depth','1','https://github.com/vectolith/velnor.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',REPO,'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard',f'origin/{BRANCH}'],check=True)
print(subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())

In [ ]:
%cd /content/velnor
!python -m pip install --upgrade pip
!python -m pip install -r training/requirements-gpu.txt
!python - <<'PY'
import torch
assert torch.cuda.is_available(), 'CUDA disappeared after dependency installation'
print('CUDA remains available:', torch.cuda.get_device_name(0))
PY

## Run full training
This uses the complete training and validation pools. It does not set `VELNOR_FAST_TRAIN=1`. The trainer selects BF16 only when the GPU supports it; otherwise it uses FP16.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil, subprocess
from pathlib import Path
run_root = Path('/content/drive/MyDrive/VELNOR-training-runs')
run_root.mkdir(parents=True, exist_ok=True)
run_name = 'run-' + subprocess.check_output(['date','-u','+%Y%m%dT%H%M%SZ'],text=True).strip()
RUN_DIR = run_root / run_name
RUN_DIR.mkdir()
outputs = Path('/content/velnor/training/outputs')
if outputs.exists() or outputs.is_symlink():
    shutil.rmtree(outputs) if outputs.is_dir() and not outputs.is_symlink() else outputs.unlink()
(RUN_DIR / 'outputs').mkdir()
outputs.symlink_to(RUN_DIR / 'outputs', target_is_directory=True)
print('Training checkpoints will be written directly to:', RUN_DIR)


In [ ]:
%%bash
set -euo pipefail
cd /content/velnor/training
python -u train_velnor_lora.py 2>&1 | tee /content/velnor-training.log
test -s outputs/velnor-coder-v2/adapter/adapter_config.json
test -s pilot_report.json
echo 'Training and adapter existence checks passed.'

In [ ]:
import hashlib, json, shutil, zipfile
from pathlib import Path
adapter = Path('/content/velnor/training/outputs/velnor-coder-v2/adapter')
report = Path('/content/velnor/training/pilot_report.json')
shutil.copy2(report, RUN_DIR / 'pilot_report.json')
shutil.copy2('/content/velnor-training.log', RUN_DIR / 'training.log')
archive = RUN_DIR / 'velnor-coder-v2-adapter.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in adapter.rglob('*'):
        if p.is_file(): z.write(p, p.relative_to(adapter.parent))
manifest = {}
for p in [archive, RUN_DIR/'pilot_report.json', RUN_DIR/'training.log']:
    h=hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    manifest[p.name]=h.hexdigest()
(RUN_DIR/'SHA256SUMS.json').write_text(json.dumps(manifest,indent=2))
print('Saved verified run artifacts to:', RUN_DIR)
print(json.dumps(manifest,indent=2))
